# YuvaIntern Week 1: Wine Dataset Data Acquisition, Cleaning and Preprocessing

**Objective:** Acquire a public dataset, inspect its quality, handle missing/duplicate/invalid values, detect and handle outliers, and document the preprocessing decisions.

**Dataset:** UCI Wine Recognition dataset.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
df = wine.frame.rename(columns={"target": "wine_class"})
df.head()


In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())


## Initial data quality findings

The dataset contains 178 rows and 13 numeric predictors plus the target class. The initial inspection checks missing values, duplicate records, and basic descriptive statistics before any transformation.


In [ ]:
df.describe().T


In [ ]:
feature_cols = [c for c in df.columns if c != "wine_class"]

outlier_counts = {}
bounds = {}

for col in feature_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    bounds[col] = (lower, upper)
    outlier_counts[col] = int(((df[col] < lower) | (df[col] > upper)).sum())

pd.Series(outlier_counts).sort_values(ascending=False)


## Outlier treatment

Outliers are identified with the 1.5 × IQR rule. Instead of deleting observations, the project caps values outside the lower/upper IQR fences. This preserves the number of observations while reducing the influence of extreme values on later statistical analysis.


In [ ]:
clean = df.copy()

for col in feature_cols:
    lower, upper = bounds[col]
    clean[col] = clean[col].clip(lower=lower, upper=upper)

clean.describe().T


In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
clean["wine_class"].value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_title("Wine Class Distribution")
ax.set_xlabel("Wine class")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
clean.to_csv("wine_cleaned.csv", index=False)
print("Saved cleaned dataset:", clean.shape)
